# 04e — Features de intensidad (6 features)


In [1]:
import pandas as pd
import numpy as np
import re, time
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase4_gustos' else Path.cwd()
DATA_RAW = ROOT / 'data' / 'data_raw'
DATA_QC_GUSTOS = ROOT / 'data' / 'data_qc_gustos'
INFORMES = ROOT / 'informes' / 'fase4_gustos' / '02_features'
DATA_QC_GUSTOS.mkdir(parents=True, exist_ok=True)

REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')

OID_RE = re.compile(r'[0-9a-f]{24}')
def clean_id(s):
    if pd.isna(s): return None
    m = OID_RE.search(str(s))
    return m.group(0) if m else None

sample = pd.read_parquet(DATA_QC_GUSTOS / 'sample_user_ids_gustos.parquet')
sample_ids = set(sample['user_id'])
N = len(sample)
features = pd.DataFrame({'user_id': sample['user_id'].values})
print(f'Sample N: {N:,}')

import pickle
t0 = time.time()


Sample N: 114,412


In [2]:
# Mapping char→user (necesario para fights/arena)
chars = pd.read_csv(DATA_RAW/'characters.csv', usecols=['_id','user_id'], low_memory=False)
chars['char_id'] = chars['_id'].apply(clean_id)
chars['user_id'] = chars['user_id'].apply(clean_id)
chars = chars.dropna(subset=['char_id','user_id'])
char_to_user = dict(zip(chars['char_id'], chars['user_id']))
del chars
print(f'char_to_user: {len(char_to_user):,}')


char_to_user: 1,336,143


In [3]:
# I01-I03 currency
print('[currency intensity]... chunks')
t = time.time()
with open(DATA_QC_GUSTOS/'_currency_agg_total.pkl','rb') as f:
    import pickle
    n_tx = pickle.load(f)
total_volume = {}
for chunk in pd.read_csv(DATA_RAW/'currency_transactions.csv', usecols=['user_id','quantity'], chunksize=1_000_000, low_memory=False):
    chunk['user_id'] = chunk['user_id'].apply(clean_id)
    chunk = chunk[chunk['user_id'].isin(sample_ids)]
    if len(chunk)==0: continue
    for u, s in chunk.groupby('user_id')['quantity'].apply(lambda x: x.abs().sum()).items():
        total_volume[u] = total_volume.get(u,0) + s
features['currency_n_transactions_30d'] = features['user_id'].map(n_tx).fillna(0)
features['currency_total_volume_30d'] = features['user_id'].map(total_volume).fillna(0)
features['currency_avg_quantity_per_tx'] = (features['currency_total_volume_30d'] / features['currency_n_transactions_30d'].replace(0, np.nan))
print(f'OK {time.time()-t:.1f}s')


[currency intensity]... chunks


OK 5.5s


In [4]:
# I04-I05 fights
print('[fights intensity]... chunks')
t = time.time()
with open(DATA_QC_GUSTOS/'_fights_total.pkl','rb') as f:
    import pickle
    fights_total = pickle.load(f)
features['fights_total_30d'] = features['user_id'].map(fights_total).fillna(0)

# avg duration en segundos
sum_dur = {}; n_dur = {}
for chunk in pd.read_csv(DATA_RAW/'fights_log.csv', usecols=['player_id','start_time','end_time'], chunksize=500_000, low_memory=False):
    chunk['player_id'] = chunk['player_id'].apply(clean_id)
    chunk['user_id'] = chunk['player_id'].map(char_to_user)
    chunk = chunk.dropna(subset=['user_id'])
    chunk = chunk[chunk['user_id'].isin(sample_ids)]
    if len(chunk)==0: continue
    chunk['dur'] = (chunk['end_time'] - chunk['start_time']).abs()
    chunk = chunk[chunk['dur'].between(0, 3600)]  # filter outliers
    for u, g in chunk.groupby('user_id')['dur']:
        sum_dur[u] = sum_dur.get(u,0) + g.sum()
        n_dur[u] = n_dur.get(u,0) + len(g)
features['fights_avg_duration'] = features['user_id'].map({u: sum_dur.get(u,0)/n_dur.get(u,1) if n_dur.get(u,0)>0 else None for u in n_dur})
print(f'OK {time.time()-t:.1f}s')


[fights intensity]... chunks


OK 65.3s


In [5]:
# I06 arena
print('[arena]')
t = time.time()
ar = pd.read_csv(DATA_RAW/'arena_log.csv', usecols=['attacker_id','winner_id'], low_memory=False)
ar['attacker_id'] = ar['attacker_id'].apply(clean_id)
ar['winner_id'] = ar['winner_id'].apply(clean_id)
ar['user_id'] = ar['attacker_id'].map(char_to_user)
ar = ar.dropna(subset=['user_id'])
ar = ar[ar['user_id'].isin(sample_ids)]

arena_total = ar.groupby('user_id').size()
features['arena_total_combats_30d'] = features['user_id'].map(arena_total.to_dict()).fillna(0).astype(int)
# guardar para 04f
ar.to_parquet(DATA_QC_GUSTOS / '_arena_sample.parquet', index=False)
print(f'OK {time.time()-t:.1f}s, users con arena: {len(arena_total):,}')


[arena]


OK 1.5s, users con arena: 7,172


In [6]:
assert len(features)==N and features['user_id'].is_unique
features.to_parquet(DATA_QC_GUSTOS / 'features_intensidad.parquet', index=False)
print(f'Guardado: {features.shape}')
nan_rates = (features.isna().mean()*100).round(2)
print(nan_rates.sort_values(ascending=False).to_string())
rep_dir = INFORMES / '04e_intensidad'
rep_dir.mkdir(parents=True, exist_ok=True)
report = [f'# 04e - Intensidad\n', f'**Shape**: {features.shape}\n', f'**Tiempo**: {time.time()-t0:.1f}s\n', '\n| Feature | %NaN |', '|---|---:|']
for f, v in nan_rates.sort_values(ascending=False).items():
    report.append(f'| {f} | {v} |')
(rep_dir / 'execution_report.md').write_text('\n'.join(report))


Guardado: (114412, 7)
fights_avg_duration             82.01
currency_avg_quantity_per_tx    81.34
user_id                          0.00
currency_n_transactions_30d      0.00
currency_total_volume_30d        0.00
fights_total_30d                 0.00
arena_total_combats_30d          0.00


319